v3:
1. Change to Monthly Cadence (starting 2026)
2. Add 'Month' column (ISO YYYY-MM format) to all data pipelines
3. Pre-2026 data remains quarterly (Month = None)
4. 2026+ data: monthly aggregation with forward-fill for survey scores
5. Dynamic Quarter/Month derivation (no hardcoded CASE statements for 2026+)

# Libraries

In [189]:
import pandas as pd
import numpy as np
import datetime
import glob
import os
import re

# Helper Functions - Monthly/Quarterly Logic

In [190]:
# ---- MONTHLY CADENCE CUTOVER DATE ----
MONTHLY_START_DATE = '2026-01-01'

def derive_month_from_date(date_series):
    """Derive Month (YYYY-MM) from a date column. Returns None for pre-2026 dates."""
    date_series = pd.to_datetime(date_series)
    return np.where(
        date_series >= pd.Timestamp(MONTHLY_START_DATE),
        date_series.dt.strftime('%Y-%m'),
        None
    )

def derive_quarter_from_date(date_series):
    """Derive Quarter string (e.g., Q1'26) from a date column."""
    date_series = pd.to_datetime(date_series)
    return date_series.apply(
        lambda d: f"Q{d.quarter}'{str(d.year)[2:]}" if pd.notna(d) else None
    )

def derive_quarter_from_month(month_series):
    """Derive Quarter string from Month (YYYY-MM) column."""
    def _to_quarter(m):
        if pd.isna(m) or m is None:
            return None
        year = m[:4]
        month_num = int(m[5:7])
        q = (month_num - 1) // 3 + 1
        return f"Q{q}'{year[2:]}"
    return month_series.apply(_to_quarter)

def get_groupby_keys(include_month=True):
    """Return standard groupby keys. Include Month for 2026+ data."""
    keys = ['Location', 'JobFamilyGroup', 'Quarter']
    if include_month:
        keys.append('Month')
    return keys

def get_merge_keys(include_month=True):
    """Return standard merge keys. Include Month for 2026+ data."""
    keys = ['Location', 'JobFamilyGroup', 'Quarter']
    if include_month:
        keys.append('Month')
    return keys

def forward_fill_scores(df, score_cols=['Score'], group_cols=['Location', 'JobFamilyGroup']):
    """Forward-fill survey scores for months within a quarter where no survey was conducted.
    Only applies to 2026+ monthly data."""
    df = df.sort_values(group_cols + ['Month'])
    for col in score_cols:
        df[col] = df.groupby(group_cols)[col].ffill()
    return df

def parse_period_from_filename(filename):
    """Parse Quarter or Month from filename.
    Handles both quarterly (Q1'25) and monthly (2026-01) formats."""
    basename = filename.split('_')[-1].split('.')[0]
    # Check if it's ISO month format (YYYY-MM)
    if re.match(r'\d{4}-\d{2}', basename):
        return {'Month': basename, 'Quarter': derive_quarter_from_month(pd.Series([basename])).iloc[0]}
    else:
        # Quarterly format like Q1'25
        return {'Month': None, 'Quarter': basename}

# Snowflake SQL snippets for dynamic Month/Quarter derivation
SNOWFLAKE_MONTH_QUARTER_SQL = """
    TO_CHAR(DATE_DAY, 'YYYY-MM') AS Month,
    CONCAT('Q', DATE_PART('QUARTER', DATE_DAY), '''', RIGHT(DATE_PART('YEAR', DATE_DAY), 2)) AS Quarter
"""

print('Helper functions loaded. Monthly cadence starts from:', MONTHLY_START_DATE)

Helper functions loaded. Monthly cadence starts from: 2026-01-01


# Connections

In [191]:
from carvana_sql import SQLServer # type: ignore
db_conn = SQLServer.connect('PEOPLEOPSPROD,1439')

In [192]:
cnx_str = "Driver={ODBC Driver 17 for SQL Server};Server=PEOPLEOPSPROD,1439;Database=PEOPLEOPS;Trusted_Connection=yes;"
cnx_str_app = "Driver={ODBC Driver 17 for SQL Server};Server=dwhcar-l;Database=OpsGroup;Trusted_Connection=yes;"

import pyodbc
odbcconn = pyodbc.connect(cnx_str)
odbcconnops = pyodbc.connect(cnx_str_app)
odbcquery = 'Select Top 10 * from PeopleOps.tenstreet.FactDriver'
odbcdf = pd.read_sql_query(odbcquery, odbcconn)

C:\Users\E102068\AppData\Local\Temp\ipykernel_2796\2098714686.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  odbcdf = pd.read_sql_query(odbcquery, odbcconn)


In [193]:
# Today's date
date_today=pd.to_datetime(datetime.date.today())

In [194]:
from carvana_sql import Snowflake

SnowflakeWarehouse = 'PEOPLEOPS_ANALYTICS'
SnowflakeRole = 'PEOPLEOPS_ANALYTICS'
SnowflakeEmail = 'anish.middela@carvana.com'
SnowflakeDatabase = "PEOPLEOPS_SANDBOX"
Snowflakeschema="SITERISK"

db_snowflake = Snowflake.connect(
                username=SnowflakeEmail,                
                role=SnowflakeRole,
                warehouse=SnowflakeWarehouse)

db_snowflake_write = Snowflake.connect(
                username=SnowflakeEmail,                
                role=SnowflakeRole,
                warehouse=SnowflakeWarehouse,
                database=SnowflakeDatabase,
                schema= Snowflakeschema )

# CheckPoint

### Location Files

In [195]:
# Netwrok csv pull
network_location_path="X:\People_Operations\Secured\Talent Analytics\Anish Middela\SiteRisk"

# Level 1 csv files
network_location_files= glob.glob(os.path.join(network_location_path, "**\*.csv"),recursive=True)

In [196]:
network_location_files

['X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Checkpoint\\Checkpoint_Q3_2025.csv',
 'X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Checkpoint\\Checkpoint_Q4_2025.csv',
 "X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Glint Reports\\Q1'24\\ADESA\\glint_Q1'24_ADESA.csv",
 "X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Glint Reports\\Q1'24\\Carvana\\glint_Carvana_Q1'24_Logistics.csv",
 "X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Glint Reports\\Q1'24\\Carvana\\glint_Carvana_Q1'24_Market Operations.csv",
 "X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Glint Reports\\Q1'24\\Carvana\\glint_Carvana_Q1'24_Post Production.csv",
 "X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Glint Reports\\Q1'24\\Carvana\\glint_Carvana_Q1'24_Reconditioning.csv",
 "X:\\People_Operations\\Secured\\Talent Analyt

In [197]:
glint_files = []
injury_files = []
navex_files = []
overall_files=[]
checkpoint_files=[]


for path in network_location_files:
    if "Overall" in path:
        overall_files.append(path)
    elif "Navex" in path:
        navex_files.append(path)
    elif "Checkpoint" in path:
        checkpoint_files.append(path)
    elif "Glint" in path:
        glint_files.append(path)
    elif "Injuries" in path:
        injury_files.append(path)

In [198]:
# files=os.listdir("C:/Users/E102068/Desktop/SiteRisk Dashboard/Glint Reports/Q1_24")

# # Files with overall data
# main_path="Glint Reports\Q1_24\Overall"

# main_files= glob.glob(main_path+"\*.csv")

In [199]:
navex_files

['X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Navex\\navex_latest.csv',
 "X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Navex\\navex_Q1'25.csv",
 "X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Navex\\navex_Q2'25.csv",
 "X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Navex\\navex_Q3'25.csv",
 "X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Navex\\navex_Q4'25.csv"]

In [200]:
glint_files

["X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Glint Reports\\Q1'24\\ADESA\\glint_Q1'24_ADESA.csv",
 "X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Glint Reports\\Q1'24\\Carvana\\glint_Carvana_Q1'24_Logistics.csv",
 "X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Glint Reports\\Q1'24\\Carvana\\glint_Carvana_Q1'24_Market Operations.csv",
 "X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Glint Reports\\Q1'24\\Carvana\\glint_Carvana_Q1'24_Post Production.csv",
 "X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Glint Reports\\Q1'24\\Carvana\\glint_Carvana_Q1'24_Reconditioning.csv",
 "X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Glint Reports\\Q1'24\\Carvana\\glint_Carvana_Q1'24_Regulatory Operations.csv",
 "X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Glint Reports\\Q1'24\\Carvana\\glint_Carva

In [201]:
location_df = pd.DataFrame()
# Appending all the csv files into one df
for file in glint_files:
    df_temp=pd.read_csv(file)
    df_temp['JobFamilyGroup']= file.split('_')[-1].split('.')[0]
    
    # v3: Quarter comes from folder path (same as v2), Month = None (Glint is quarterly)
    df_temp['Quarter'] = file.split('_')[-2]

    location_df=pd.concat([location_df,df_temp],ignore_index=True)

In [202]:
# Select only neccesary columns
location_df=location_df.iloc[:,[0,5,6,15,16]]

# Rename
location_df= location_df.rename(columns={location_df.columns[1]:'Score',location_df.columns[2]:'eSat Change'})

# Filter out Location ==ALL
location_df = location_df[~(location_df['Location'] == 'All')]

In [203]:
# Convert 'Score' column to numeric, errors='coerce' will replace non-numeric values with NaN
location_df['Score'] = pd.to_numeric(location_df['Score'], errors='coerce')
location_df['eSat Change'] = pd.to_numeric(location_df['eSat Change'], errors='coerce')

# Fill na == 0
location_df=location_df.fillna(0)

# Peakon / Checkpoint

In [204]:
checkpoint_files

['X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Checkpoint\\Checkpoint_Q3_2025.csv',
 'X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Checkpoint\\Checkpoint_Q4_2025.csv']

In [205]:
# Checkpoint Data Pull
# v3: Filename format is Checkpoint_Q4_2025.csv -> parse Q4 and 2025 to build Q4'25
checkpoint_df = pd.DataFrame()
for file in checkpoint_files:
    df = pd.read_csv(file)
    # Parse Quarter from filename: Checkpoint_Q4_2025.csv -> Q4'25
    parts = os.path.basename(file).replace('.csv','').split('_')
    quarter_num = parts[-2]  # e.g., 'Q4'
    year = parts[-1]         # e.g., '2025'
    df['Quarter'] = f"{quarter_num}'{year[2:]}"  # e.g., "Q4'25"
    checkpoint_df = pd.concat([checkpoint_df, df], ignore_index=True)

Peakon Data Cleaning

In [206]:
checkpoint_df.columns

Index(['Location', 'JobFamilyGroup', 'Segment', 'Participants',
       'Participation_Percentage', 'Score', 'Quarter'],
      dtype='object')

In [207]:
checkpoint_df['JobFamilyGroup'] = np.where(checkpoint_df['JobFamilyGroup'].str.contains('ADESA'), 'ADESA',checkpoint_df['JobFamilyGroup'])

In [208]:
checkpoint_df['Employee_Size_Checkpoint'] = checkpoint_df['Participants']/checkpoint_df['Participation_Percentage']
checkpoint_df['Employee_Size_Checkpoint'] = checkpoint_df['Employee_Size_Checkpoint'].round(0).astype(int)

In [209]:
checkpoint_df = checkpoint_df[['Location','JobFamilyGroup','Participants','Employee_Size_Checkpoint','Score','Quarter']]

In [210]:
# v3: Checkpoint stays quarterly - group by Location/JobFamilyGroup/Quarter only (no Month)
# Scores will be forward-filled into monthly rows later during the main_df merge
checkpoint_df = checkpoint_df.groupby(['Location','JobFamilyGroup','Quarter']).agg({'Participants':'sum','Employee_Size_Checkpoint':'sum','Score':'mean'}).reset_index()

In [211]:
checkpoint_df['Score'] = checkpoint_df['Score'].round(1)

In [212]:
## Trim whitespace from Location, JobFamilyGroup, and Quarter columns
checkpoint_df['Location'] = checkpoint_df['Location'].str.strip()
checkpoint_df['JobFamilyGroup'] = checkpoint_df['JobFamilyGroup'].str.strip()
checkpoint_df['Quarter'] = checkpoint_df['Quarter'].str.strip()

In [213]:
# Create empty columns in location_df for merging later
location_df['Participants']=None
location_df['Employee_Size_Checkpoint']=None

In [214]:
location_df=location_df[['Location','JobFamilyGroup','Participants','Employee_Size_Checkpoint','Score','Quarter']]

In [215]:
final_checkpoint_df = pd.concat([location_df, checkpoint_df], ignore_index=True)

In [216]:
final_checkpoint_df['Quarter'].value_counts()

Quarter
Q2'24    417
Q3'24    415
Q1'25    405
Q2'25    405
Q1'24    400
Q4'25    399
Q4'24    398
Q3'25    358
Name: count, dtype: int64

### Necessary JobFamilyGroups

In [217]:
necessary_JobFamilyGroups = ['Logistics','Market Operations','Post Production',
                             'Reconditioning','Regulatory Operations','Wholesale Operations','ADESA','Registration']

In [218]:
final_checkpoint_df = final_checkpoint_df[final_checkpoint_df['JobFamilyGroup'].isin(necessary_JobFamilyGroups)]

In [219]:
# necessary_JobFamilyGroups= location_df['JobFamilyGroup'].unique().tolist()

In [220]:
# Get value counts for 'JobFamilyGroup' in Q4'25 quarter
final_checkpoint_df[final_checkpoint_df['Quarter'] == "Q4'25"]['JobFamilyGroup'].value_counts()

JobFamilyGroup
Market Operations       126
ADESA                    60
Logistics                40
Post Production          36
Reconditioning           35
Registration             26
Wholesale Operations      2
Name: count, dtype: int64

In [221]:
# Compare value counts for each quarter side by side
# This will create a DataFrame with JobFamilyGroup as rows and Quarters as columns
value_counts_df = final_checkpoint_df.groupby(['Quarter', 'JobFamilyGroup']).size().unstack(fill_value=0)
value_counts_df

JobFamilyGroup,ADESA,Logistics,Market Operations,Post Production,Reconditioning,Registration,Regulatory Operations,Wholesale Operations
Quarter,,,,,,,,
Q1'24,95,49,145,19,24,0,44,24
Q1'25,94,49,146,30,33,0,39,14
Q2'24,96,49,152,22,29,0,44,25
Q2'25,94,49,146,30,33,0,39,14
Q3'24,95,47,149,24,31,0,46,23
Q3'25,59,38,103,36,35,17,0,2
Q4'24,94,47,146,25,28,0,39,19
Q4'25,60,40,126,36,35,26,0,2


In [222]:
final_checkpoint_df['JobFamilyGroup'] = np.where(final_checkpoint_df['JobFamilyGroup'].str.contains('Regulatory'), 'Registration', final_checkpoint_df['JobFamilyGroup'])

# v3: Derive DATE_QUARTER timestamp from Quarter string (e.g., Q4'25 -> 2025-10-01)
def quarter_str_to_timestamp(q):
    """Convert Quarter string like Q4'25 to timestamp 2025-10-01."""
    if pd.isna(q) or q is None:
        return pd.NaT
    match = re.match(r"Q([1-4])'(\d{2})", str(q))
    if match:
        qtr = int(match.group(1))
        year = 2000 + int(match.group(2))
        month = (qtr - 1) * 3 + 1  # Q1->1, Q2->4, Q3->7, Q4->10
        return pd.Timestamp(year=year, month=month, day=1)
    return pd.NaT

final_checkpoint_df['DATE_QUARTER'] = final_checkpoint_df['Quarter'].apply(quarter_str_to_timestamp)

# v3: Forward-fill checkpoint/survey scores across months
# Logic: Quarterly checkpoint data (Month=None) gets forward-filled into monthly rows.
# Example: Q4'25 score fills into 2026-01, 2026-02, 2026-03 until Q1'26 data overwrites it.
# Sort by Quarter then Month so quarterly (None) rows come before monthly rows
final_checkpoint_df = final_checkpoint_df.sort_values(['Location', 'JobFamilyGroup', 'Quarter'])
for col in ['Score', 'Participants', 'Employee_Size_Checkpoint']:
    final_checkpoint_df[col] = final_checkpoint_df.groupby(['Location', 'JobFamilyGroup'])[col].ffill()

# Injuries

In [223]:
# Old Logic
# injury_path=("Injuries")

# injury_files= glob.glob(os.path.join(injury_path, "*.csv"))

# injuries_df= glob.glob(injury_path+'\*.csv')
# injuries_df=pd.read_csv("C:/Users/E102068/Desktop/SiteRisk Dashboard/Injuries/injuries_report_Q1'24.csv")

In [224]:
injury_files

["X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Injuries\\injuries_report_Q1'24.csv",
 "X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Injuries\\injuries_report_Q1'25.csv",
 "X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Injuries\\injuries_report_Q2'24.csv",
 "X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Injuries\\injuries_report_Q3'24.csv",
 "X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Injuries\\injuries_report_Q4'24.csv"]

In [225]:
injury_df_main= pd.DataFrame()

# Appending all the csv files into one DataFrame
for file in injury_files:
    df_temp_injury = pd.read_csv(file)


    # Strip leading and trailing spaces from column names
    df_temp_injury.columns = df_temp_injury.columns.str.strip()


    # Add Quarter information
    df_temp_injury['Quarter'] = file.split('_')[-1].split('.')[0]


    # Concatenate to main DataFrame
    injury_df_main = pd.concat([injury_df_main, df_temp_injury], ignore_index=True)


In [226]:
# v3: Updated Snowflake injury query - pre-2026 uses CASE, 2026+ uses dynamic Month/Quarter
snowflake_pull_df = db_snowflake.execute(
    """
SELECT CLAIM_NUMBER as CASE_NUMBER,
       OCCURRENCE_NUMBER,
       STATUS,
       OCCURRED_DAY,
       C_FIRSTDAYOFMONTHDATE,
       LOCATION_NAME as WORK_LOCATION,
       EMPLOYEE_ID_C as EMPLOYEE_ID,
       TITLE as JOB_PROFILE,
       DEPARTMENT,
       JOB_FAMILY_GROUP,
       JOB_FAMILY,
       BODY_PART as BODYPART2,
       DATE_TRUNC('QUARTER', OCCURRED_DAY) AS DATE_QUARTER,
       DATE_TRUNC('MONTH', OCCURRED_DAY) AS DATE_MONTH,
       CONCAT('Q', DATE_PART('QUARTER', OCCURRED_DAY), '''', RIGHT(DATE_PART('YEAR', OCCURRED_DAY), 2)) AS Quarter
FROM CARLEGAL.WC_INJURY.WC_CLAIMS_INPUT
WHERE OCCURRED_DAY >= '2025-04-01';

""")

c:\Users\E102068\AppData\Local\Programs\Python\Python311\Lib\site-packages\snowflake\connector\config_manager.py:351: UserWarning: Bad owner or permissions on C:\Users\E102068\.snowflake\connections.toml
  warn(f"Bad owner or permissions on {str(filep)}{chmod_message}")


Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...
Going to open: https://carvana.okta.com/app/snowflake/exkhr8majsTrZPBMF1t7/sso/saml?SAMLRequest=jVJdb%2BIwEPwrke%2BZ2CSUA4tQpUXRoSu9qIQ%2B8GYSAy6OHbwOofz6c%2Fg49R5a9c1az%2BzM7uzo%2FlhK78ANCK0i1PUJ8rjKdSHUJkKLLOkMkAeWqYJJrXiE3jmg%2B%2FEIWCkrGtd2q174vuZgPddIAW0%2FIlQbRTUDAVSxkgO1OZ3Hsyca%2BIQyAG6sk0NXSgHCaW2trSjGTdP4Tehrs8EBIQSTIXaoFvIDfZCovtaojLY61%2FJGObqZPpHoYtJrJRzCKaRX4oNQlxV8pbK6gID%2ByrK0k%2F6ZZ8iLb9M9agV1yc2cm4PI%2BeLl6WIAnIPfcdwbhkHog9LNWrIdz3VZ1dY1890Lr3mBpd4It6LpJELVThT7QajL02a9OsTrfMX22algyV4OTgs7W6Yy6SevoSa7zWo4y5H3egs0aAOdAtR8qtoYrSuRoN8hYYf0syCgdyHt9f07EiyRN3ExCsXsmXnzmjNzYIr5emfZ2RyrKvzPN%2BbH3dYMSvYGmVmmD7Oka39iAI3blNDlUOjZgBl%2Fc%2FwR%2Fki6ntqz2%2F50kmop8ncv0aZk9vNwun73XBFFZ32GUl4yIeOiMBzAhSSlbh4NZ9ZdtDU1R3h8Uf3%2Fpsd%2FAQ%3D%3D&RelayState=

In [227]:
snowflake_pull_df= snowflake_pull_df.rename(columns={
    'QUARTER': 'Quarter',
    'JOB_FAMILY_GROUP': 'JobFamilyGroup',
    'JOB_PROFILE': 'JobProfile', 
    'JOB_FAMILY': 'JobFamily'})

In [228]:
injury_df_main = pd.concat([injury_df_main, snowflake_pull_df], ignore_index=True)

In [229]:
injury_df_main.groupby('Quarter').size()

Quarter
Q1'24    346
Q1'25    534
Q1'26    491
Q2'24    398
Q2'25    497
Q3'24    447
Q3'25    557
Q4'24    419
Q4'25    630
dtype: int64

In [230]:
injury_df_main['JobFamilyGroup'].unique()

array([nan, 'RECONDITIONING', 'MARKET_OPERATIONS', 'JOB_FAMILY-3-381',
       'POST_PRODUCTION', 'LOGISTICS', 'ACCOUNTING', 'JOB_FAMILY-3-379',
       'JOB_FAMILY-3-380', 'JOB_FAMILY-3-382', 'JOB_FAMILY-3-383',
       'Operations', 'SAFE_AND_SECURE', 'JOB_FAMILY-3-375', 'INVENTORY',
       'REGULATORY_OPERATIONS', 'Reconditioning', 'Market Operations',
       'Post Production', 'Logistics', 'Reconditioning Services (adesa)',
       'Safe And Secure', 'Operations (adesa)', 'Regulatory Operations',
       'Legal', 'Wholesale Operations', 'Sell To Carvana',
       'Safe And Secure (adesa)', 'Technology Services', 'Verification',
       'Sales (adesa)', 'Customer Service Group (adesa)',
       'Customer Experience', 'Engineering', 'Treasury Operations',
       'People Operations', 'Dealership', 'JOB_FAMILY-3-470',
       'JOB_FAMILY-3-494', 'CUSTOMER_EXPERIENCE', 'WHOLESALE_OPERATIONS',
       'JOB_FAMILY-3-451', 'JOB_FAMILY-3-471', 'JOB_FAMILY-3-500',
       'STC_CUSTOMER_CARE', 'TECHNOLO

In [231]:
# Transform the 'JobFamilyGroup' column
injury_df_main['JobFamilyGroup'] = injury_df_main['JobFamilyGroup'].apply(
    lambda x: x.upper().replace(' ', '_') if isinstance(x, str) else x
)

In [232]:
# Pull JobProfile Detail from workday - TABLE EMPLOYEE HISTORY
JobFamily_df = db_snowflake.execute("""
                             
SELECT Distinct JOB_PROFILE,JOB_FAMILY_GROUP 
FROM PEOPLEOPS.ANALYTICS.VW_WORKDAY_EMPLOYEE_LINEAGE
WHERE COMPANY NOT like '%ADESA%'
AND JOB_FAMILY_GROUP NOT like '%ADESA%';

""")

In [233]:
JobFamily_df=JobFamily_df.rename(columns={'JOB_PROFILE':'JobProfile','JOB_FAMILY_GROUP':'JobFamilyGroup'})

JobFamily_df['JobFamilyGroup'] = np.where(JobFamily_df['JobFamilyGroup'].str.contains('Regulatory'), 'Registration', JobFamily_df['JobFamilyGroup'])

JobFamily_df = JobFamily_df.drop_duplicates()

In [234]:
# Merging with Workday Jobfamily to get more accurate information
injury_df_main= pd.merge(injury_df_main,JobFamily_df, how='left', on='JobProfile')

# injury_df_check= pd.merge(injury_df,JobFamily_df, how='left', left_on='JobProfile',right_on='JobProfileName')

In [235]:
distinct_empl= db_snowflake.execute("""

WITH RankedEmployees AS (
    SELECT 
        DATE_DAY, 
        EMPLOYEE_ID, 
        LOCATION,
        ROW_NUMBER() OVER (PARTITION BY EMPLOYEE_ID ORDER BY DATE_DAY DESC) as rn
    FROM 
        PEOPLEOPS.ANALYTICS.VW_WORKDAY_EMPLOYEE_LINEAGE
    WHERE 
        LOCATION IS NOT NULL
)

SELECT  
    EMPLOYEE_ID, 
    LOCATION                   
FROM 
    RankedEmployees
WHERE 
    rn = 1
ORDER BY 
    EMPLOYEE_ID;

""")

In [236]:
distinct_empl['EMPLOYEE_ID'] = distinct_empl['EMPLOYEE_ID'].astype(int)
injury_df_main['EMPLOYEE_ID'] = injury_df_main['EMPLOYEE_ID'].fillna(0).astype(int)

In [237]:
injury_df_main=pd.merge(injury_df_main,distinct_empl,on='EMPLOYEE_ID',how='left')

In [238]:
injury_df_main['WORK_LOCATION']= np.where(injury_df_main['LOCATION'].isna(),injury_df_main['WORK_LOCATION'],injury_df_main['LOCATION'])

In [239]:
print(necessary_JobFamilyGroups)
print(injury_df_main['JobFamilyGroup_x'].unique())

['Logistics', 'Market Operations', 'Post Production', 'Reconditioning', 'Regulatory Operations', 'Wholesale Operations', 'ADESA', 'Registration']
[nan 'RECONDITIONING' 'MARKET_OPERATIONS' 'JOB_FAMILY-3-381'
 'POST_PRODUCTION' 'LOGISTICS' 'ACCOUNTING' 'JOB_FAMILY-3-379'
 'JOB_FAMILY-3-380' 'JOB_FAMILY-3-382' 'JOB_FAMILY-3-383' 'OPERATIONS'
 'SAFE_AND_SECURE' 'JOB_FAMILY-3-375' 'INVENTORY' 'REGULATORY_OPERATIONS'
 'RECONDITIONING_SERVICES_(ADESA)' 'OPERATIONS_(ADESA)' 'LEGAL'
 'WHOLESALE_OPERATIONS' 'SELL_TO_CARVANA' 'SAFE_AND_SECURE_(ADESA)'
 'TECHNOLOGY_SERVICES' 'VERIFICATION' 'SALES_(ADESA)'
 'CUSTOMER_SERVICE_GROUP_(ADESA)' 'CUSTOMER_EXPERIENCE' 'ENGINEERING'
 'TREASURY_OPERATIONS' 'PEOPLE_OPERATIONS' 'DEALERSHIP' 'JOB_FAMILY-3-470'
 'JOB_FAMILY-3-494' 'JOB_FAMILY-3-451' 'JOB_FAMILY-3-471'
 'JOB_FAMILY-3-500' 'STC_CUSTOMER_CARE' 'INFRASTRUCTURE_DEVELOPMENT'
 'AUCTION' 'TRANSPORTATION' 'PHOTOBOOTH' 'BODY_SHOP' 'MECHANICAL'
 'MAINTENANCE' 'DETAIL' 'OFFICE_OPERATIONS' None 'REGISTRATIO

In [240]:
# Renaming the column
injury_df_main=injury_df_main.rename(columns={'JobFamilyGroup_y':'JobFamilyGroup'})


In [241]:
# Wherever their are NA then pointing it to Company
injury_df_main['JobFamilyGroup'] = np.where(injury_df_main['JobFamilyGroup'].isna(),injury_df_main['JobFamilyGroup_x'],injury_df_main['JobFamilyGroup'])

In [242]:
injury_df_main['JobFamilyGroup'].unique()

array([nan, 'Reconditioning', 'Market Operations', 'JOB_FAMILY-3-381',
       'Post Production', 'Logistics', 'ACCOUNTING', 'JOB_FAMILY-3-379',
       'JOB_FAMILY-3-380', 'JOB_FAMILY-3-382', 'JOB_FAMILY-3-383',
       'OPERATIONS', 'SAFE_AND_SECURE', 'JOB_FAMILY-3-375',
       'Registration', 'Titles', 'Inventory', 'Safe and Secure',
       'RECONDITIONING', 'MARKET_OPERATIONS', 'POST_PRODUCTION',
       'RECONDITIONING_SERVICES_(ADESA)', 'LOGISTICS',
       'OPERATIONS_(ADESA)', 'REGULATORY_OPERATIONS', 'Legal',
       'Wholesale Operations', 'SELL_TO_CARVANA', 'WHOLESALE_OPERATIONS',
       'SAFE_AND_SECURE_(ADESA)', 'TECHNOLOGY_SERVICES', 'Verification',
       'SALES_(ADESA)', 'CUSTOMER_SERVICE_GROUP_(ADESA)', 'VERIFICATION',
       'CUSTOMER_EXPERIENCE', 'ENGINEERING', 'TREASURY_OPERATIONS',
       'People Operations', 'Dealership', 'JOB_FAMILY-3-470',
       'JOB_FAMILY-3-494', 'Customer Experience', 'JOB_FAMILY-3-451',
       'JOB_FAMILY-3-471', 'JOB_FAMILY-3-500', 'Sell to Carv

In [243]:
print(necessary_JobFamilyGroups)

['Logistics', 'Market Operations', 'Post Production', 'Reconditioning', 'Regulatory Operations', 'Wholesale Operations', 'ADESA', 'Registration']


In [244]:
# Define the conditions for JobRole
conditions = [
    injury_df_main['JobFamilyGroup'].str.contains('ADESA|adesa', regex=True, na=False),
    injury_df_main['JobFamilyGroup'].str.contains('RECONDITIONING|Inventory', regex=True, na=False),
    injury_df_main['JobFamilyGroup'].str.contains('WHOLESALE_OPERATIONS', regex=True, na=False),
    injury_df_main['JobFamilyGroup'].str.contains('POST_PRODUCTION', regex=True, na=False),
    injury_df_main['JobFamilyGroup'].str.contains('REGULATORY_OPERATIONS|Registration', regex=True, na=False),
    injury_df_main['JobFamilyGroup'].str.contains('LOGISTICS|TRANSPORTATION', regex=True, na=False),
    injury_df_main['JobFamilyGroup'].str.contains('MARKET_OPERATIONS', regex=True, na=False),
]


# Define the corresponding JobRole values
# values = ['ADESA','Reconditioning','Wholesale Operations','Post Production','Regulatory Operations','Logistics','Market Operations']
values = ['ADESA','Reconditioning','Wholesale Operations','Post Production','Registration','Logistics','Market Operations']


# Use np.select to create the JobRole column based on the conditions and values
injury_df_main['JobFamilyGroup'] = np.select(conditions, values, injury_df_main['JobFamilyGroup'])

In [245]:
injury_df_main['JobFamilyGroup'] = np.where(injury_df_main['JobFamilyGroup'].isna(),injury_df_main['COMPANY'],injury_df_main['JobFamilyGroup'])

# Convert DATE_QUARTER and DATE_MONTH to timestamp, keep null as null
injury_df_main['DATE_QUARTER'] = pd.to_datetime(injury_df_main['DATE_QUARTER'], errors='coerce')
injury_df_main['DATE_MONTH'] = pd.to_datetime(injury_df_main['DATE_MONTH'], errors='coerce')

# Renaming Col
injury_df_snowflake= injury_df_main.rename(columns={'WORK_LOCATION':'Location','BODYPART2':'Injuries'})

### Snowflake - Injury

In [246]:
injury_df_snowflake = injury_df_snowflake[['Injuries','Location','JobFamilyGroup','JobProfile','Quarter','COMPANY','STATUS','CASE_NUMBER','DATE_QUARTER','DATE_MONTH']]

In [247]:
injury_df_snowflake.head()

,Injuries,Location,JobFamilyGroup,JobProfile,Quarter,COMPANY,STATUS,CASE_NUMBER,DATE_QUARTER,DATE_MONTH
0,Back,ADESA Riverside,ADESA,NaN,Q1'24,ADESA,submitted,ADESA-7740,NaT,NaT
1,Back,Salt Lake,ADESA,NaN,Q1'24,ADESA,completed,ADESA-7708,NaT,NaT
2,"['Shoulder(s)', 'Ankle']",Los Angeles,ADESA,NaN,Q1'24,ADESA,completed,ADESA-7668,NaT,NaT
3,Knee,ADESA Cincinnati,ADESA,NaN,Q1'24,ADESA,completed,ADESA-7719,NaT,NaT
4,Finger(s),"Belton, MO",ADESA,NaN,Q1'24,ADESA,completed,ADESA-7530,NaT,NaT


In [248]:
injury_df_snowflake.groupby('Quarter').size()

Quarter
Q1'24    348
Q1'25    534
Q1'26    491
Q2'24    398
Q2'25    497
Q3'24    447
Q3'25    557
Q4'24    419
Q4'25    634
dtype: int64

In [249]:
# # Uncomment when you want to run the script again
# Write into Snowflake
db_snowflake_write.insert('injuries',injury_df_snowflake,truncate=True)

c:\Users\E102068\AppData\Local\Programs\Python\Python311\Lib\site-packages\snowflake\connector\config_manager.py:351: UserWarning: Bad owner or permissions on C:\Users\E102068\.snowflake\connections.toml
  warn(f"Bad owner or permissions on {str(filep)}{chmod_message}")


Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...
Going to open: https://carvana.okta.com/app/snowflake/exkhr8majsTrZPBMF1t7/sso/saml?SAMLRequest=jVJdc9owEPwrHvUZS7YhwRog44TxlOajHjCZhjdhC1CwJVcnY%2Fj3lQ100odk%2BqY57d7u3d7o7lgWzoFrEEqOkecS5HCZqVzI7Rgt07g3RA4YJnNWKMnH6MQB3U1GwMqiolFtdnLOf9ccjGMbSaDtxxjVWlLFQACVrORATUYX0fMT9V1CGQDXxsqhCyUHYbV2xlQU46Zp3CZwld5inxCCSYgtqoV8Qx8kqq81Kq2MylRxpRztTJ9IeJj0WwmLsArJhXgv5HkFX6mszyCg39M06SU%2FFylyout0D0pCXXK94PogMr6cP50NgHXwGEX9MPADF6RqNgXb80yVVW1sM9e%2B8IbnuFBbYVc0m45RtRf5yn8P4dcrWx%2BHs%2BV68Bj%2FmIrVfB2Ht2%2FDxUllx6pfS%2B8tSCqSIef1GqjfBjoDqPlMtjEaWyL%2BTY8EPXKT%2Bj4dBHQQur4XrpAztTEKyUzHvHrNmD4wyVy1N6wzx6oK%2F%2FWN%2BXG%2F08OSvUOqV8n9c%2ByZWwygcJsSOh8K7QzoyX%2BOP8IfSZdTe7Hbn00TVYjs5MRKl8x8Ho7nel1F5L1NB6W8ZKKI8lxzABtSUajmQXNm7EUbXXOEJ2fVf2968gc%3D&RelayState=ver%3A3-hi

Uploading data chunk num: 1: 100%|██████████| 4325/4325 [00:00<00:00, 5474.83it/s]


True

### Injury df

In [250]:
# Filtering Only for necessary_JobFamilyGroups
injury_df= injury_df_snowflake[injury_df_snowflake['JobFamilyGroup'].isin(necessary_JobFamilyGroups)]

In [251]:
injury_df['JobFamilyGroup'].unique()

array(['ADESA', 'Reconditioning', 'Market Operations', 'Post Production',
       'Logistics', 'Registration', 'Wholesale Operations'], dtype=object)

In [252]:
# Counting the number of injuries at the Location-Org Combination
# v3: Include Month in groupby for monthly granularity
injury_df = injury_df.groupby(['Location','JobFamilyGroup','Quarter','DATE_QUARTER','DATE_MONTH']).agg('count').reset_index()

# Final Injury df
injury_df = injury_df[['Location', 'JobFamilyGroup','Quarter','Injuries','DATE_QUARTER','DATE_MONTH']]


## Main Dataframe

In [253]:
# v3: Merging for Specific CostCentre - include Month in merge keys
# main_df= pd.merge(injury_df,location_df, how='outer', on=['Location','JobFamilyGroup','Quarter','DATE_QUARTER'])

In [254]:
main_df = injury_df [['Location', 'JobFamilyGroup','Quarter','DATE_QUARTER','DATE_MONTH','Injuries']]

# Performance PIPs/CARs

In [255]:
## Snowflake Pull for Performance Data

performance_snowflake_base_df = db_snowflake.execute("""
                                                     
with cte as (
SELECT *
FROM PEOPLEOPS.WORKDAY.VW_EMPLOYEE_REVIEWS
WHERE REVIEW_CATEGORY IN ('Corrective Action' ,'Performance Improvement Plan')
AND DATE_INITIATED::DATE>='2024-01-01'
AND DATE_INITIATED::DATE <= LAST_DAY(DATEADD(month, -1, DATE_TRUNC(month, CURRENT_DATE)),'month')
)

SELECT c.EMPLOYEE_ID
,c.REVIEW_CATEGORY
,c.REVIEW_REASON
,c.RESULTING_ACTION_RATING
,c.DATE_INITIATED
,c.STATUS
,eh.START_TIME
,eh.END_TIME
,eh.JOB_FAMILY_GROUP
,eh.COMPANY
,eh.LOCATION_NAME AS LOCATION
,CONCAT('Q',DATE_PART('QUARTER',c.DATE_INITIATED),'''',RIGHT(DATE_PART('YEAR',C.DATE_INITIATED),2)) AS QUARTER
,DATE_TRUNC('QUARTER', c.DATE_INITIATED) AS DATE_QUARTER
,DATE_TRUNC('MONTH', c.DATE_INITIATED) AS DATE_MONTH
FROM CTE c
LEFT JOIN PEOPLEOPS.WORKDAY.VW_DIM_EMPLOYEE_HISTORY eh 
ON c.EMPLOYEE_ID = eh.EMPLOYEE_ID
WHERE c.DATE_INITIATED BETWEEN eh.START_TIME AND eh.END_TIME
ORDER BY C.DATE_INITIATED;
                                                     
""")

In [256]:
performance_snowflake_base_df.head()

,EMPLOYEE_ID,REVIEW_CATEGORY,REVIEW_REASON,RESULTING_ACTION_RATING,DATE_INITIATED,STATUS,START_TIME,END_TIME,JOB_FAMILY_GROUP,COMPANY,LOCATION,QUARTER,DATE_QUARTER,DATE_MONTH
0,94657,Corrective Action,Attendance (United States of America),Verbal Warning,2024-01-02 03:18:05.688,Successfully Completed,2023-11-20 05:26:08,2024-04-17 20:42:00,Reconditioning,"Carvana, LLC",GA Winder IC,Q1'24,2024-01-01,2024-01-01
1,97414,Corrective Action,Attendance (United States of America),Termination,2024-01-02 03:23:37.656,Successfully Completed,2023-11-28 06:15:40,2024-01-03 09:33:29,Reconditioning,"Carvana, LLC",NC Concord IC,Q1'24,2024-01-01,2024-01-01
2,99322,Corrective Action,Attendance (United States of America),Termination,2024-01-02 05:02:06.346,Successfully Completed,2023-12-25 08:24:42,2024-01-11 10:34:55,Logistics,"Carvana Logistics, LLC",NJ Delanco IC,Q1'24,2024-01-01,2024-01-01
3,32950,Corrective Action,Attendance (United States of America),Written Warning,2024-01-02 05:07:18.574,Successfully Completed,2023-05-31 04:26:15,2024-04-16 20:37:19,Logistics,"Carvana Logistics, LLC",AL Bessemer IC,Q1'24,2024-01-01,2024-01-01
4,97493,Corrective Action,Attendance (United States of America),Written Warning,2024-01-02 05:19:05.466,Canceled,2023-10-23 02:58:50,2024-01-04 09:54:32,Reconditioning Services,ADESA US Auction LLC,FL Orlando - Sanford,Q1'24,2024-01-01,2024-01-01


### In progress performance 

In [257]:
performance_inprogress_snowflake_base_df = db_snowflake.execute(f"""
SELECT EMPLOYEE_ID
,MANAGER
,REVIEW_TEMPLATE
,PROCESS_STATUS as STATUS
,PROCESS_INITIATION_DATE as DATE_INITIATED
,STEP
,STEP_STATUS
,STEP_COMPLETED_ON
,STEP_COMPLETED_BY
FROM PEOPLEOPS.WORKDAY.VW_EMPLOYEE_REVIEW_PROCESS
WHERE PROCESS_INITIATION_DATE  >='2024-01-01'
AND PROCESS_INITIATION_DATE <= LAST_DAY(DATEADD(month, -1, DATE_TRUNC(month, CURRENT_DATE)),'month')
AND STEP LIKE '%To Do%'
AND STEP_STATUS ='Step Completed'
AND PROCESS_STATUS = 'In Progress'
ORDER BY PROCESS_INITIATION_DATE desc;
""")

In [258]:
# performance_sql_df.value_counts('Status')

In [259]:
performance_snowflake_base_df[performance_snowflake_base_df['QUARTER']=="Q1'26"].value_counts('STATUS')

STATUS
Successfully Completed    3813
In Progress                536
Canceled                   456
Rescinded                    8
Name: count, dtype: int64

In [260]:
# Creating Keys to check if the value exist in inprogess df (can be done using merge)

# performance_sql_df['key']=performance_sql_df['EmployeeId'].astype(str)+performance_sql_df['Status'].astype(str)\
#     +performance_sql_df['DateInitiated'].astype(str)
# performance_inprogress_sql_df['key']=performance_inprogress_sql_df['EmployeeId'].astype(str)+performance_inprogress_sql_df['Status'].astype(str)\
#     +performance_inprogress_sql_df['DateInitiated'].astype(str)

# # Finding rows that exist in inprogress and updating their status
# performance_sql_df.loc[performance_sql_df['key'].isin(performance_inprogress_sql_df['key']), 'Status'] = 'Successfully Completed'


performance_snowflake_base_df['key']=performance_snowflake_base_df['EMPLOYEE_ID'].astype(str)+performance_snowflake_base_df['STATUS'].astype(str)\
    +performance_snowflake_base_df['DATE_INITIATED'].astype(str)
performance_inprogress_snowflake_base_df['key']=performance_inprogress_snowflake_base_df['EMPLOYEE_ID'].astype(str)+performance_inprogress_snowflake_base_df['STATUS'].astype(str)\
    +performance_inprogress_snowflake_base_df['DATE_INITIATED'].astype(str)

# Finding rows that exist in inprogress and updating their status
performance_snowflake_base_df.loc[performance_snowflake_base_df['key'].isin(performance_inprogress_snowflake_base_df['key']), 'STATUS'] = 'Successfully Completed'

In [261]:
# drop the key column
performance_snowflake_base_df = performance_snowflake_base_df.drop(columns=['key'])
performance_inprogress_snowflake_base_df = performance_inprogress_snowflake_base_df.drop(columns=['key'])

In [262]:
performance_snowflake_base_df.value_counts('STATUS')

STATUS
Successfully Completed    53122
Canceled                   6504
In Progress                 338
Rescinded                    98
Name: count, dtype: int64

### ADESA PIPs/CARs

In [263]:
# Converting Job Families in company ADESA as ADESA 
# performance_sql_df['JobFamilyGroup'] = np.where(performance_sql_df['Company'].str.contains('ADESA'),"ADESA",performance_sql_df['JobFamilyGroup'])

performance_snowflake_base_df['JOB_FAMILY_GROUP'] = np.where(performance_snowflake_base_df['COMPANY'].str.contains('ADESA'),"ADESA",performance_snowflake_base_df['JOB_FAMILY_GROUP'])


In [264]:
# Neccessary Job Family Group
# Filtering for siterisk costcentres
performance_snowflake_base_df=performance_snowflake_base_df[performance_snowflake_base_df['JOB_FAMILY_GROUP'].isin(necessary_JobFamilyGroups)]

In [265]:
performance_snowflake_base_df.value_counts('STATUS')

STATUS
Successfully Completed    50748
Canceled                   6032
In Progress                 323
Rescinded                    87
Name: count, dtype: int64

### Snowflake - Performance 

In [266]:
performance_snowflake_df = performance_snowflake_base_df.copy()

In [267]:
performance_snowflake_df.head()

,EMPLOYEE_ID,REVIEW_CATEGORY,REVIEW_REASON,RESULTING_ACTION_RATING,DATE_INITIATED,STATUS,START_TIME,END_TIME,JOB_FAMILY_GROUP,COMPANY,LOCATION,QUARTER,DATE_QUARTER,DATE_MONTH
0,94657,Corrective Action,Attendance (United States of America),Verbal Warning,2024-01-02 03:18:05.688,Successfully Completed,2023-11-20 05:26:08,2024-04-17 20:42:00,Reconditioning,"Carvana, LLC",GA Winder IC,Q1'24,2024-01-01,2024-01-01
1,97414,Corrective Action,Attendance (United States of America),Termination,2024-01-02 03:23:37.656,Successfully Completed,2023-11-28 06:15:40,2024-01-03 09:33:29,Reconditioning,"Carvana, LLC",NC Concord IC,Q1'24,2024-01-01,2024-01-01
2,99322,Corrective Action,Attendance (United States of America),Termination,2024-01-02 05:02:06.346,Successfully Completed,2023-12-25 08:24:42,2024-01-11 10:34:55,Logistics,"Carvana Logistics, LLC",NJ Delanco IC,Q1'24,2024-01-01,2024-01-01
3,32950,Corrective Action,Attendance (United States of America),Written Warning,2024-01-02 05:07:18.574,Successfully Completed,2023-05-31 04:26:15,2024-04-16 20:37:19,Logistics,"Carvana Logistics, LLC",AL Bessemer IC,Q1'24,2024-01-01,2024-01-01
4,97493,Corrective Action,Attendance (United States of America),Written Warning,2024-01-02 05:19:05.466,Canceled,2023-10-23 02:58:50,2024-01-04 09:54:32,ADESA,ADESA US Auction LLC,FL Orlando - Sanford,Q1'24,2024-01-01,2024-01-01


In [268]:
performance_snowflake_df.groupby('QUARTER').size()  

QUARTER
Q1'24    3997
Q1'25    6922
Q1'26    4536
Q2'24    4752
Q2'25    7401
Q3'24    5317
Q3'25    9325
Q4'24    5498
Q4'25    9442
dtype: int64

In [269]:
performance_snowflake_df.describe()

,DATE_INITIATED,START_TIME,END_TIME,DATE_QUARTER,DATE_MONTH
count,57190,57190,57190,57190,57190
mean,2025-03-26 22:02:32.859246336,2025-02-17 09:17:34.654642432,2025-04-20 11:18:42.945899776,2025-02-10 19:12:34.747333632,2025-03-11 19:41:06.431194368
min,2024-01-02 03:18:05.688000,2021-09-13 22:06:15,2024-01-02 09:23:25,2024-01-01 00:00:00,2024-01-01 00:00:00
25%,2024-10-03 11:38:33.452999936,2024-09-12 11:56:28,2024-10-10 15:00:07,2024-10-01 00:00:00,2024-10-01 00:00:00
50%,2025-04-27 09:37:57.851500032,2025-03-26 16:38:44,2025-05-25 15:47:30.500000,2025-04-01 00:00:00,2025-04-01 00:00:00
75%,2025-09-28 14:44:06.817750016,2025-08-11 07:13:16,2025-10-26 00:17:05,2025-07-01 00:00:00,2025-09-01 00:00:00
max,2026-02-28 18:43:23.921000,2026-02-28 07:17:40,2026-03-05 07:18:47,2026-01-01 00:00:00,2026-02-01 00:00:00


In [270]:
performance_snowflake_df['JOB_FAMILY_GROUP'] = np.where(performance_snowflake_df['JOB_FAMILY_GROUP'].str.contains('Regulatory'), 'Registration', performance_snowflake_df['JOB_FAMILY_GROUP'])

In [271]:
# Uncomment when you need to run it again
# Write into Snowflake
# db_snowflake_write.drop_table('Performance')
# db_snowflake_write.dump('Performance',performance_snowflake_df)
db_snowflake_write.insert('Performance',performance_snowflake_df,truncate=True)

Uploading data chunk num: 1: 100%|██████████| 57190/57190 [00:01<00:00, 50025.40it/s]


True

### Performance Df

In [272]:
performance_df= db_snowflake.execute("""

SELECT * FROM PEOPLEOPS_SANDBOX.SITERISK.PERFORMANCE

""")

In [273]:
# Filtering only for Successfully completed
performance_df=performance_df[performance_df['STATUS']=='Successfully Completed']

In [274]:
performance_df.value_counts('STATUS')

STATUS
Successfully Completed    50748
Name: count, dtype: int64

In [275]:
performance_df['QUARTER'].unique()

array(["Q1'24", "Q2'24", "Q3'24", "Q4'24", "Q1'25", "Q2'25", "Q3'25",
       "Q4'25", "Q1'26"], dtype=object)

In [276]:
# Selecting neccesary columns
# performance_df= performance_df.iloc[:,[0,1,-4,-2,-1]]
performance_df= performance_df[['EMPLOYEE_ID','LOCATION','JOB_FAMILY_GROUP','REVIEW_CATEGORY','QUARTER','DATE_QUARTER','DATE_MONTH']]

In [277]:
# Count the number of PIPs/Cars and Unique Employees
performance_df= performance_df.groupby(['LOCATION','JOB_FAMILY_GROUP','REVIEW_CATEGORY','QUARTER','DATE_QUARTER','DATE_MONTH']).\
                                agg(Total_Count=('EMPLOYEE_ID','count'),EmployeesPIPs_CARs=('EMPLOYEE_ID','nunique')).reset_index()

In [278]:
performance_df['Total_Count'].sum()

50748

In [279]:
# Pivot to divide the PIPs/CARs
performance_df= performance_df.pivot(index=['LOCATION','JOB_FAMILY_GROUP','QUARTER','DATE_QUARTER','DATE_MONTH','EmployeesPIPs_CARs'],\
                                     columns=['REVIEW_CATEGORY'],values='Total_Count').reset_index()

In [280]:
# Total sum of all
performance_df= performance_df.groupby(['LOCATION','JOB_FAMILY_GROUP','QUARTER','DATE_QUARTER','DATE_MONTH']).agg('sum').reset_index()

## Overall PIPs/CARs

In [281]:
# Filling all NA with 0 for addition
performance_df=performance_df.fillna(0)

# Sum of ALL PIPs/CARs
performance_df['PIPs & CARs']=performance_df['Performance Improvement Plan']\
                                    +performance_df['Corrective Action']

# Rename
performance_df=performance_df.rename(columns={'Performance Improvement Plan':'PIPs',\
                                              'Corrective Action':'CARs'})

In [282]:
performance_df['PIPs & CARs'].sum()

50748.0

In [283]:
#Rename Colmns
performance_df=performance_df.rename(columns={'LOCATION':'Location','JOB_FAMILY_GROUP':'JobFamilyGroup','QUARTER':'Quarter'})

In [284]:
performance_df['JobFamilyGroup'].unique()

array(['Logistics', 'Market Operations', 'Post Production',
       'Reconditioning', 'Registration', 'ADESA', 'Wholesale Operations'],
      dtype=object)

## Merge-Main Df

In [285]:
# Merge with main df
main_df=pd.merge(main_df,performance_df,on=['Location','JobFamilyGroup','Quarter','DATE_QUARTER','DATE_MONTH'],how='outer')

In [286]:
main_df.head()

,Location,JobFamilyGroup,Quarter,DATE_QUARTER,DATE_MONTH,Injuries,EmployeesPIPs_CARs,CARs,PIPs,PIPs & CARs
0,AL Bessemer IC,Logistics,Q1'24,2024-01-01,2024-01-01,NaN,10.0,15.0,0.0,15.0
1,AL Bessemer IC,Logistics,Q1'24,2024-01-01,2024-02-01,NaN,9.0,11.0,0.0,11.0
2,AL Bessemer IC,Logistics,Q1'24,2024-01-01,2024-03-01,NaN,8.0,12.0,0.0,12.0
3,AL Bessemer IC,Logistics,Q1'25,2025-01-01,2025-01-01,NaN,7.0,9.0,0.0,9.0
4,AL Bessemer IC,Logistics,Q1'25,2025-01-01,2025-02-01,NaN,10.0,15.0,0.0,15.0


# Turnover

In [287]:
turnover_snow_terms= db_snowflake_write.execute(f"""
SELECT 
    DATE_DAY as TerminationDate,
    EMPLOYEE_ID,
    EMPLOYEE_NAME as NAME,
    CASE
    WHEN JOB_FAMILY_GROUP like '%ADESA%' THEN 'ADESA'
    ELSE JOB_FAMILY_GROUP
    END as JOB_FAMILY_GROUP, 
    LOCATION, 
    CONCAT('Q', DATE_PART('QUARTER', DATE_DAY), '''', RIGHT(DATE_PART('YEAR', DATE_DAY), 2)) AS Quarter,
    DATE_TRUNC('QUARTER', DATE_DAY) AS DATE_QUARTER,
    DATE_TRUNC('MONTH', DATE_DAY) AS DATE_MONTH
FROM 
    PEOPLEOPS.ANALYTICS.VW_WORKDAY_EMPLOYEE_LINEAGE
WHERE 
    DATA_TYPE = 'TERMINATED'
    AND EMPLOYEE_TYPE='Regular'
    AND DATE_DAY>='2024-01-01'

ORDER BY 
    TerminationDate

""")

In [288]:
turnover_snow_terms['JOB_FAMILY_GROUP'].unique()

array(['Reconditioning Services', 'Verification', 'Operations',
       'Market Operations', 'Reconditioning', 'Logistics',
       'Post Production', 'Safe and Secure', 'Regulatory Operations',
       'Finance', 'Transportation', 'People Operations',
       'Product Management', 'Customer Service Group',
       'Wholesale Operations', 'Sales', None, 'Accounting',
       'Customer Experience', 'Engineering', 'Sell to Carvana',
       'Analytics', 'Facility Services', 'Block Clerk', 'Inventory',
       'Legal', 'Technology Services', 'ADESA', 'Auctioneer',
       'Treasury Operations', 'Infrastructure Development', 'Brand',
       'Special Projects', 'Titles', 'Marketplace', 'Dealership',
       'Registration'], dtype=object)

In [289]:
turnover_snow_terms['JOB_FAMILY_GROUP'] = np.where(turnover_snow_terms['JOB_FAMILY_GROUP'].str.contains('Regulatory'), 'Registration', turnover_snow_terms['JOB_FAMILY_GROUP'])

In [290]:
# Converting to Date
turnover_snow_terms['TERMINATIONDATE']=pd.to_datetime(turnover_snow_terms['TERMINATIONDATE'])

# Neccesary JobFamilyGroups
turnover_snow_terms=turnover_snow_terms[turnover_snow_terms['JOB_FAMILY_GROUP'].isin(necessary_JobFamilyGroups) & turnover_snow_terms['QUARTER'].notna()]

# # Adding Quarter based on Date column
# turnover_snow_terms['Quarter'] = turnover_snow_terms['TerminationDate'].dt.to_period("Q")
# turnover_snow_terms['Quarter'] = turnover_snow_terms['Quarter'].apply(lambda q: f"{q.strftime('Q%q')}'{str(q.year)[2:]}")

In [291]:
print(turnover_snow_terms.shape)
turnover_snow_terms.head()

(27476, 8)


,TERMINATIONDATE,EMPLOYEE_ID,NAME,JOB_FAMILY_GROUP,LOCATION,QUARTER,DATE_QUARTER,DATE_MONTH
3,2024-01-02,96081,Shanice McDaniel,Market Operations,NC Concord IC,Q1'24,2024-01-01,2024-01-01
4,2024-01-02,97893,Iyanna Edwards,Reconditioning,NC Concord IC,Q1'24,2024-01-01,2024-01-01
5,2024-01-02,95494,Histo Louis,Market Operations,MA Boston - Norfolk,Q1'24,2024-01-01,2024-01-01
6,2024-01-02,22922,Amariah Lopez-Garcia,Logistics,HQ-5,Q1'24,2024-01-01,2024-01-01
7,2024-01-02,99058,Addie Guy,Logistics,AL Bessemer IC,Q1'24,2024-01-01,2024-01-01


In [292]:
turnover_snow_terms.groupby('QUARTER').size()

QUARTER
Q1'24    1678
Q1'25    3171
Q1'26    3223
Q2'24    2068
Q2'25    3268
Q3'24    2756
Q3'25    4425
Q4'24    2563
Q4'25    4324
dtype: int64

#### Active Emp Calculation

In [293]:
# v3: Updated - pre-2026 uses CASE, 2026+ uses dynamic Month/Quarter
active_df=db_snowflake.execute("""

SELECT
    DATE_DAY as DATE,
    CASE
    WHEN JOB_FAMILY_GROUP like '%ADESA%' THEN 'ADESA'
    WHEN JOB_FAMILY_GROUP like '%Regulatory%' THEN 'Registration'
    ELSE JOB_FAMILY_GROUP
    END as JOB_FAMILY_GROUP,
    CONCAT('Q', DATE_PART('QUARTER', DATE_DAY), '''', RIGHT(DATE_PART('YEAR', DATE_DAY), 2)) AS Quarter,
    DATE_TRUNC('QUARTER', DATE_DAY) AS DATE_QUARTER,
    DATE_TRUNC('MONTH', DATE_DAY) AS DATE_MONTH,
    LOCATION,
    COUNT(*) as Active_employees
FROM 
    PEOPLEOPS.ANALYTICS.VW_WORKDAY_EMPLOYEE_LINEAGE
WHERE 
    DATA_TYPE in('ACTIVE','ON LEAVE')
    AND EMPLOYEE_TYPE='Regular'
    AND DATE_DAY>='2024-01-01'
GROUP BY DATE_DAY,LOCATION,
CASE
    WHEN JOB_FAMILY_GROUP like '%ADESA%' THEN 'ADESA'
    WHEN JOB_FAMILY_GROUP like '%Regulatory%' THEN 'Registration'
    ELSE JOB_FAMILY_GROUP END

ORDER BY DATE,JOB_FAMILY_GROUP,LOCATION,Active_employees;

""")

In [294]:
active_df['JOB_FAMILY_GROUP'] = np.where(active_df['JOB_FAMILY_GROUP'].str.contains('Regulatory'), 'Registration', active_df['JOB_FAMILY_GROUP'])

In [295]:
active_df=active_df[active_df['JOB_FAMILY_GROUP'].isin(necessary_JobFamilyGroups) & active_df['QUARTER'].notna()]

In [296]:
# # Quarter calculation
# # Calculating the average headcount for the Quater
# # (Add Week Number in groupby for weekly turnover)

# active_qt_df=active_df.groupby(['LOCATION','JOB_FAMILY_GROUP','DATE_QUARTER','DATE_MONTH'])\
#     .agg(Avg_emp=('ACTIVE_EMPLOYEES','mean')).reset_index()

# # Rounding the weekly
# active_qt_df['Avg_emp']=round(active_qt_df['Avg_emp'],2)

In [297]:
# month

active_month_df=active_df.groupby(['LOCATION','JOB_FAMILY_GROUP','QUARTER','DATE_QUARTER','DATE_MONTH'])\
    .agg(Avg_emp=('ACTIVE_EMPLOYEES','mean')).reset_index()

# Rounding the monthly
active_month_df['Avg_emp']=round(active_month_df['Avg_emp'],2)

#### Term Calculation

In [298]:
# Count number of terms each day
term_df= turnover_snow_terms.groupby(['TERMINATIONDATE','JOB_FAMILY_GROUP','LOCATION','QUARTER','DATE_QUARTER','DATE_MONTH'])\
    .agg(Terminations=('EMPLOYEE_ID','count')).reset_index()

# # Renaming Termination Date
# term_df=term_df.rename(columns={'TERMINATIONDATE':'Date'})

# # Adding Weeknumber based on Termination Date
# term_df['WeekNumber']=term_df['Date'].dt.isocalendar().week # Add this for weekly turnover

# # Adding Quarter based on Date column
# term_df['Quarter'] = term_df['Date'].dt.to_period("Q")
# term_df['Quarter'] = term_df['Quarter'].apply(lambda q: f"{q.strftime('Q%q')}'{str(q.year)[2:]}")

In [299]:
# Quarter calculation
# Calculating the terms for the quarter

term_month_df=term_df.groupby(['LOCATION','JOB_FAMILY_GROUP','QUARTER','DATE_QUARTER','DATE_MONTH'])\
    .agg(Terminations=('Terminations','sum')).reset_index()

#### Turnover

In [300]:
# # Combining Active and Terms 
# turnover_df=pd.merge(active_df,term_df,on=['Date','Quarter','Location','JobFamilyGroup'],how='left')

# # Fill NA
# turnover_df=turnover_df.fillna(0)

#### Monthly Turnover

In [301]:
# Combining Active and Terms 
turnover_month_df=pd.merge(active_month_df,term_month_df,on=['LOCATION','JOB_FAMILY_GROUP','QUARTER','DATE_QUARTER','DATE_MONTH'],how='left')

# Fill NA
turnover_month_df=turnover_month_df.fillna(0)

In [302]:
turnover_month_df= turnover_month_df.groupby(['LOCATION','JOB_FAMILY_GROUP','QUARTER','DATE_QUARTER','DATE_MONTH']).\
                            agg(Avg_emp=('Avg_emp','mean'),\
                                Terminations=('Terminations','sum')).reset_index()

# Calculating the turnover rate 
turnover_month_df['Monthly_Turnover']=round((turnover_month_df['Terminations']/turnover_month_df['Avg_emp'])*100,2)

In [303]:
turnover_month_df=turnover_month_df.rename(columns={'QUARTER':'Quarter','JOB_FAMILY_GROUP':'JobFamilyGroup','LOCATION':'Location'})

In [304]:
turnover_month_df['DATE_QUARTER'] = pd.to_datetime(turnover_month_df['DATE_QUARTER'], errors='coerce')
turnover_month_df['DATE_MONTH'] = pd.to_datetime(turnover_month_df['DATE_MONTH'], errors='coerce')

## Merge - Main DF

In [305]:
main_df=pd.merge(main_df,turnover_month_df,on=['Location','JobFamilyGroup','Quarter','DATE_QUARTER','DATE_MONTH'],how='outer')

# Navex - Siterisk 2.0

In [306]:
## SITE RISK 2.0
navex_files

['X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Navex\\navex_latest.csv',
 "X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Navex\\navex_Q1'25.csv",
 "X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Navex\\navex_Q2'25.csv",
 "X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Navex\\navex_Q3'25.csv",
 "X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Navex\\navex_Q4'25.csv"]

In [321]:
valid_locations = main_df[['Location', 'JobFamilyGroup']].drop_duplicates().to_dict('records')
# valid_locations = main_df[['Location', 'JobFamilyGroup']].drop_duplicates().to_dict('records')


print(valid_locations[:5]) # Print the first 5 to check the structure

[{'Location': 'AL Bessemer IC', 'JobFamilyGroup': 'Logistics'}, {'Location': 'AL Bessemer IC', 'JobFamilyGroup': 'Market Operations'}, {'Location': 'AL Bessemer IC', 'JobFamilyGroup': 'Post Production'}, {'Location': 'AL Bessemer IC', 'JobFamilyGroup': 'Reconditioning'}, {'Location': 'AL Bessemer IC', 'JobFamilyGroup': 'Registration'}]


In [309]:
## Select on the files with the "latest" in the name - this is the one with the most recent data
navex_files = [f for f in navex_files if 'latest' in os.path.basename(f).lower()]

print("Navex files being processed:", navex_files)

Navex files being processed: ['X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Navex\\navex_latest.csv']


In [316]:
# v3: Single Navex file - derive Quarter/Month from MONTH_OPENED column
navex_df_main = pd.read_csv(navex_files[0])  # Assuming only one "latest" file exists

# Strip leading and trailing spaces from column names
navex_df_main.columns = navex_df_main.columns.str.strip()

# Parse MONTH_OPENED (format: MM/DD/YYYY) to derive Quarter and Month
navex_df_main['DATE_OPENED'] = pd.to_datetime(navex_df_main['DATE_OPENED'], format='%m/%d/%Y')
navex_df_main['DATE_QUARTER'] = navex_df_main['DATE_OPENED'].dt.to_period('Q').dt.to_timestamp()
navex_df_main['DATE_MONTH'] = navex_df_main['DATE_OPENED'].dt.to_period('M').dt.to_timestamp()
navex_df_main['Quarter'] = derive_quarter_from_date(navex_df_main['DATE_OPENED'])

In [317]:
# navex_df_main= navex_df_main.drop(columns=['LATITUTDE','LONGITUDE'])
navex_df_main=navex_df_main.drop_duplicates()
navex_df_main=navex_df_main.rename(columns={'BRANCH_NUMBER':'JobFamilyGroup'})

In [318]:
navex_df_main['JobFamilyGroup'].unique()

array(['Market Operations', nan, 'Reconditioning', 'Post Production',
       'Logistics', 'People Operations', 'Wholesale', 'Transportation',
       'Safe and Secure', 'ADESA Customer Care', 'Auction',
       'ADESA Auction', 'Office Operations', 'Inventory', 'ADESA',
       'Photobooth', 'Registration', 'Verification',
       'Accounting and Capital Markets', 'Corporate',
       'Customer Experience', 'ADESA Safe & Secure', 'Engineering',
       'Adesa Auction', 'Detail', 'Regulatory Operations',
       'Technology Services', 'Finance Operations', 'Unknown',
       'Lot Operations', 'ADESA Customer Relations Sales',
       'Wholesale Operations', 'Titles', 'Build', 'Talent Acquisition',
       'Title & Registration', 'Inspection Center Development',
       'Strategy and Analytics', 'Cosmetic', 'STC Customer Care',
       'Adesa Inspection', 'Safe & Secure', 'Sell to Carvana',
       'Auction Operations'], dtype=object)

In [319]:
navex_df_main['JobFamilyGroup'] = np.where(navex_df_main['JobFamilyGroup'].str.contains('Regulatory'), 'Registration', navex_df_main['JobFamilyGroup'])

In [322]:
from fuzzywuzzy import fuzz
from fuzzywuzzy import process


def map_location(row, valid_locations, min_score=80):
    """
    Maps a given location name to the best matching valid location
    within the same job family group, prioritizing specific conditions.

    Args:
        row (pd.Series): A row from the DataFrame containing 'LOCATION_NAME' and 'JOBFAMILYGROUP'.
        valid_locations (list): A list of dictionaries with 'Location' and 'JobFamilyGroup'.
        min_score (int): Minimum fuzzy match score (0-100) to consider a match.

    Returns:
        str: The best matching valid location, or None if no good match is found.
    """
    location_name = row['LOCATION_NAME']
    job_family = row['JobFamilyGroup']

    if pd.isna(location_name):
        return None

    location_name_upper = location_name.upper()

    if location_name == 'Remote':
        return 'Remote'
    elif location_name == 'Other':
        return 'Other'
    elif 'VM' in location_name_upper or 'ADESA' in location_name_upper or 'VLC' in location_name_upper:
        return location_name
    else:
        # Filter valid locations by job family group
        relevant_locations = [
            loc['Location'] for loc in valid_locations if loc['JobFamilyGroup'] == job_family
        ]
        if not relevant_locations:
            return None  # No valid locations for this job family

        best_match, score = process.extractOne(location_name, relevant_locations, scorer=fuzz.token_set_ratio)
        if score >= min_score:
            return best_match
        return None

# Assuming your DataFrame 'df' has columns 'LOCATION_NAME' and 'JOBFAMILYGROUP'
navex_df_main['Mapped_Location'] = navex_df_main.apply(map_location, axis=1, valid_locations=valid_locations)


In [323]:
navex_df_main[['LOCATION_NAME', 'JobFamilyGroup', 'Mapped_Location']]

,LOCATION_NAME,JobFamilyGroup,Mapped_Location
0,TN Memphis - Getwell,Market Operations,TN Memphis - Getwell
1,NaN,Registration,None
2,NaN,Registration,None
3,OK Oklahoma City IC,Reconditioning,OK Oklahoma City IC
4,OK Oklahoma City IC,Reconditioning,OK Oklahoma City IC
...,...,...,...
1944,OH Heath IC,Reconditioning,OH Heath IC
1945,Other,Registration,Other
1946,PA York IC,Registration,PA York IC
1947,TX Blue Mound IC,Reconditioning,TX Blue Mound IC


In [ ]:
navex_df_main['Mapped_Location']= np.where(navex_df_main['Mapped_Location'].isna(),navex_df_main['LOCATION_NAME'],navex_df_main['Mapped_Location'])

In [ ]:
navex_df_main.head()

In [ ]:
navex_df_main.value_counts('Quarter')

In [ ]:
navex_df_main.columns

In [ ]:
navex_df_main=navex_df_main.rename(columns={'LOCATION_NAME':'Old_Location','Mapped_Location':'Location'})

navex_df_main['JobFamilyGroup'].unique()

In [ ]:
# navex_df_main= navex_df_main.drop(columns=['QUARTER'])

## Snowflake

In [ ]:
# Snowflake Dump
navex_df_snowflake= navex_df_main.copy()
db_snowflake_write.drop_table('Navex')
db_snowflake_write.dump('Navex',navex_df_snowflake)

## Navex Df

In [ ]:
navex_df= navex_df_main.copy()
navex_df= navex_df[navex_df['JobFamilyGroup'].isin(necessary_JobFamilyGroups)]

In [ ]:
# v3: Include Month in groupby for monthly granularity
if 'Month' not in navex_df.columns:
    navex_df['Month'] = None
navex_df= navex_df.groupby(['Location','JobFamilyGroup','Quarter','Month'])\
    .agg(Navex_Incidents=('CASE_NUMBER','nunique')).reset_index()
    # .agg(Navex_Incidents=('CASE_STATUS','count')).reset_index()

In [ ]:
navex_df.value_counts('Quarter')

In [ ]:
navex_df['Navex_Incidents'].sum()

In [ ]:
navex_df.head()

In [ ]:
navex_df['JobFamilyGroup'].unique()

## Main df

In [ ]:
# v3: Include Month in Navex merge
main_df=pd.merge(main_df,navex_df,on=['Location','JobFamilyGroup','Quarter','Month'],how='left')
# main_df['Navex_Incidents']=main_df['Navex_Incidents'].fillna(0)

# Total Employees- Location/JobProfile

*** ADD QUARTER DATA IN 'CASE' STATEMENT

In [ ]:
# v3: Updated - pre-2026 uses day 89 of quarter, 2026+ uses last day of month
totalemp_Loc_Jbp_df = db_snowflake.execute("""

-- Pre-2026: quarterly snapshot (day 89 of quarter)
SELECT
    CASE
    WHEN JOB_FAMILY_GROUP like '%ADESA%' THEN 'ADESA'
    WHEN JOB_FAMILY_GROUP like '%Regulatory%' THEN 'Registration'
    ELSE JOB_FAMILY_GROUP
    END as JOB_FAMILY_GROUP,
    CASE
        WHEN DATE_QUARTER = '2024-01-01' THEN 'Q1''24'
        WHEN DATE_QUARTER = '2024-04-01' THEN 'Q2''24'
        WHEN DATE_QUARTER = '2024-07-01' THEN 'Q3''24'
        WHEN DATE_QUARTER = '2024-10-01' THEN 'Q4''24'
        WHEN DATE_QUARTER = '2025-01-01' THEN 'Q1''25'                                                                           
        WHEN DATE_QUARTER = '2025-04-01' THEN 'Q2''25'
        WHEN DATE_QUARTER = '2025-07-01' THEN 'Q3''25'
        WHEN DATE_QUARTER = '2025-10-01' THEN 'Q4''25'                                                                   
    END AS Quarter,
    NULL AS Month,
    LOCATION,
    COUNT(*) as Total_employees
FROM 
    PEOPLEOPS.ANALYTICS.VW_WORKDAY_EMPLOYEE_LINEAGE
WHERE 
    DATA_TYPE = 'ACTIVE'
    AND EMPLOYEE_TYPE='Regular'
    AND DATEDIFF(DAY,DATE_QUARTER,DATE_DAY)=89
    AND DATE_DAY>='2024-01-01'
    AND DATE_DAY<'2026-01-01'
GROUP BY DATE_DAY,LOCATION,DATE_QUARTER,
CASE
    WHEN JOB_FAMILY_GROUP like '%ADESA%' THEN 'ADESA'
    WHEN JOB_FAMILY_GROUP like '%Regulatory%' THEN 'Registration'
    ELSE JOB_FAMILY_GROUP END

UNION ALL

-- v3: 2026+ monthly snapshot (last day of each month)
SELECT
    CASE
    WHEN JOB_FAMILY_GROUP like '%ADESA%' THEN 'ADESA'
    WHEN JOB_FAMILY_GROUP like '%Regulatory%' THEN 'Registration'
    ELSE JOB_FAMILY_GROUP
    END as JOB_FAMILY_GROUP,
    CONCAT('Q', DATE_PART('QUARTER', DATE_DAY), '''', RIGHT(DATE_PART('YEAR', DATE_DAY), 2)) AS Quarter,
    TO_CHAR(DATE_DAY, 'YYYY-MM') AS Month,
    LOCATION,
    COUNT(*) as Total_employees
FROM 
    PEOPLEOPS.ANALYTICS.VW_WORKDAY_EMPLOYEE_LINEAGE
WHERE 
    DATA_TYPE = 'ACTIVE'
    AND EMPLOYEE_TYPE='Regular'
    AND DATE_DAY = LAST_DAY(DATE_DAY, 'month')
    AND DATE_DAY>='2026-01-01'
GROUP BY DATE_DAY,LOCATION,
CASE
    WHEN JOB_FAMILY_GROUP like '%ADESA%' THEN 'ADESA'
    WHEN JOB_FAMILY_GROUP like '%Regulatory%' THEN 'Registration'
    ELSE JOB_FAMILY_GROUP END

ORDER BY Quarter,JOB_FAMILY_GROUP,LOCATION,Total_employees;

""")

In [ ]:
totalemp_Loc_Jbp_df=totalemp_Loc_Jbp_df.drop_duplicates()

In [ ]:
totalemp_Loc_Jbp_df=totalemp_Loc_Jbp_df[totalemp_Loc_Jbp_df['JOB_FAMILY_GROUP'].isin(necessary_JobFamilyGroups) & totalemp_Loc_Jbp_df['QUARTER'].notna()]

totalemp_Loc_Jbp_df=totalemp_Loc_Jbp_df.rename(columns={'LOCATION':'Location','JOB_FAMILY_GROUP':'JobFamilyGroup','QUARTER':'Quarter','TOTAL_EMPLOYEES':'Total_Employees'})

In [ ]:
totalemp_Loc_Jbp_df.groupby('Quarter').size()

In [ ]:
totalemp_Loc_Jbp_df['JobFamilyGroup'].unique()

In [ ]:
# total_emp_ADESA= db_snowflake.execute("""

# SELECT 'ADESA' as JOB_FAMILY_GROUP,LOCATION,
#     CASE
#         WHEN DATE_QUARTER = '2024-01-01' THEN 'Q1''24'
#         WHEN DATE_QUARTER = '2024-04-01' THEN 'Q2''24'
#         WHEN DATE_QUARTER = '2024-07-01' THEN 'Q3''24'
#         ELSE 'Unknown'
#     END AS Quarter
#     ,Count(*) as Total_Employees
# FROM PEOPLEOPS_SANDBOX.DEV.WORKDAY_EMPLOYEE_LINEAGE
# WHERE DATA_TYPE='ACTIVE'
# AND 
# AND DATEDIFF(DAY,DATE_QUARTER,DATE_DAY)=90
# AND DATE_DAY>='2024-01-01'
# AND JOB_PROFILE Like '%ADESA%'
# GROUP BY LOCATION,DATE_QUARTER
# ORDER BY Total_Employees desc,
# LOCATION;

# """)

In [ ]:
# totalemp_Loc_Jbp_df=pd.concat([totalemp_Loc_Jbp_df,total_emp_ADESA])

## Active Qt Avg Emp

In [ ]:
# active_qt_df=active_qt_df.rename(columns={'LOCATION':'Location','JOB_FAMILY_GROUP':'JobFamilyGroup','QUARTER':'Quarter','Avg_emp':'Average_Headcount'})

## Merge- Main df

In [ ]:
# v3: Add Month to totalemp before merge
if 'Month' not in totalemp_Loc_Jbp_df.columns:
    totalemp_Loc_Jbp_df['Month'] = None
main_df= pd.merge(main_df,totalemp_Loc_Jbp_df,on=['Location','JobFamilyGroup','Quarter','Month'],how='left')

In [ ]:
main_df['Location']=main_df['Location'].astype(str)

In [ ]:
# main_df=main_df.drop(columns=['DATE'])

In [ ]:
# main_df=pd.merge(main_df,active_qt_df,on=['Location','JobFamilyGroup','Quarter'],how='left')

In [ ]:
main_df['PIPs & CARs'].sum()

# Leadership

In [ ]:
# v3: Updated - pre-2026 uses CASE, 2026+ uses dynamic Month/Quarter
leadership = db_snowflake.execute(""" 
with cte as (
SELECT DATE_WEEK,LOCATION,JOB_FAMILY_GROUP
,SUM(TOTAL_LEADERS) as TOTAL_LEADERS
,SUM(NEW_LEADERS) as NEW_LEADERS
,SUM(LEADER_HIRED) as LEADER_HIRED
,SUM(TERMINATIONS) as LEADER_TERMINATIONS
,SUM(OPENINGS) as OPENINGS
FROM PEOPLEOPS_SANDBOX.SITERISK.LEADERSHIP
GROUP BY ALL
)
,
-- Pre-2026: quarterly with CASE statements
pre_2026 as (
SELECT
DATE_TRUNC('quarter',DATE_WEEK) as DATE_QUARTER
,    CASE
        WHEN DATE_QUARTER = '2025-01-01' THEN 'Q1''25'                                                                           
        WHEN DATE_QUARTER = '2025-04-01' THEN 'Q2''25'
        WHEN DATE_QUARTER = '2025-07-01' THEN 'Q3''25'
        WHEN DATE_QUARTER = '2025-10-01' THEN 'Q4''25'                                                                          
    END AS Quarter
, NULL AS Month
,LOCATION
,JOB_FAMILY_GROUP
,COALESCE(ROUND(AVG(TOTAL_LEADERS),0),0) as AVG_TOTAL_LEADERS
,COALESCE(ROUND(AVG(NEW_LEADERS),0),0) as AVG_NEW_LEADERS
,COALESCE(ROUND(AVG(OPENINGS),0),0) as AVG_OPENINGS
, ROUND(DIV0(AVG_NEW_LEADERS,AVG_TOTAL_LEADERS)*100,1) as PERCENTAGE_NEW_LEADER
, ROUND(DIV0(AVG_TOTAL_LEADERS,(AVG_TOTAL_LEADERS + AVG_OPENINGS))*100,1) as PERCENTAGE_STAFFED
FROM cte
WHERE DATE_WEEK < '2026-01-01'
AND QUARTER IS NOT NULL
GROUP BY ALL
)
,
-- v3: 2026+ monthly data with dynamic Month/Quarter
post_2026 as (
SELECT
DATE_TRUNC('quarter',DATE_WEEK) as DATE_QUARTER
, CONCAT('Q', DATE_PART('QUARTER', DATE_WEEK), '''', RIGHT(DATE_PART('YEAR', DATE_WEEK), 2)) AS Quarter
, TO_CHAR(DATE_WEEK, 'YYYY-MM') AS Month
,LOCATION
,JOB_FAMILY_GROUP
,COALESCE(ROUND(AVG(TOTAL_LEADERS),0),0) as AVG_TOTAL_LEADERS
,COALESCE(ROUND(AVG(NEW_LEADERS),0),0) as AVG_NEW_LEADERS
,COALESCE(ROUND(AVG(OPENINGS),0),0) as AVG_OPENINGS
, ROUND(DIV0(AVG_NEW_LEADERS,AVG_TOTAL_LEADERS)*100,1) as PERCENTAGE_NEW_LEADER
, ROUND(DIV0(AVG_TOTAL_LEADERS,(AVG_TOTAL_LEADERS + AVG_OPENINGS))*100,1) as PERCENTAGE_STAFFED
FROM cte
WHERE DATE_WEEK >= '2026-01-01'
GROUP BY ALL
)
SELECT * FROM pre_2026
UNION ALL
SELECT * FROM post_2026;
                                   """)

In [ ]:
leadership.head()

In [ ]:
leadership.rename(columns={'LOCATION':'Location','JOB_FAMILY_GROUP':'JobFamilyGroup','QUARTER':'Quarter'},inplace=True)

In [ ]:
leadership.columns

In [ ]:
# v3: Add Month to leadership before merge
if 'Month' not in leadership.columns:
    leadership['Month'] = None
main_df=pd.merge(main_df,leadership[['Location','JobFamilyGroup','Quarter','Month','AVG_TOTAL_LEADERS', 'AVG_NEW_LEADERS', 'AVG_OPENINGS','PERCENTAGE_NEW_LEADER', 'PERCENTAGE_STAFFED']]\
                 ,on=['Location','JobFamilyGroup','Quarter','Month'],how='left')

In [ ]:
# main_df.rename(columns={'AVG_LEADERS':'Avg_Leaders', 'LEADER_HIRED':'Leader_Hired', 'LEADER_TERMED':'Leader_Termed'}, inplace=True)

# Overtime

In [ ]:
# overtime_snow = db_snowflake.execute("""
#     SELECT 
#         LOCATIONNAME as LOCATION
#         ,DATE_TRUNC('quarter',DATE) AS DATE_QUARTER
#         ,CASE
#         WHEN DATE_QUARTER = '2025-01-01' THEN 'Q1''25'                                                                           
#         WHEN DATE_QUARTER = '2025-04-01' THEN 'Q2''25'
#         WHEN DATE_QUARTER = '2025-07-01' THEN 'Q3''25'
#         WHEN DATE_QUARTER = '2025-10-01' THEN 'Q4''25'                                                                          
#         END AS Quarter
#         ,'Reconditioning' AS JOB_FAMILY_GROUP
#         ,ROUND(SUM(OVERTIMEHOURS),0) as TOTAL_OVERTIME
#     FROM PEOPLEOPS_SANDBOX.DEV.IC_OVERTIME_AVG_HOURS
#     WHERE DATE>='2025-01-01'
#     GROUP BY LOCATIONNAME,DATE_TRUNC('quarter',DATE);
#                                      """)


### UPDATE TO SNOWFLAKE VERSION TABLE [INVENTORY.PUBLIC.EXPECTED_VS_PAID_HOURS]

In [ ]:
# v3: Updated - pre-2026 uses CASE, 2026+ uses dynamic Month/Quarter (SQL Server syntax)
overtimequery = """Select LocationName as LOCATION,
DATETRUNC(QUARTER,Date) as DATE_QUARTER,
CASE 
 WHEN JobFamilyGroup like '%RECON%' Then 'Reconditioning'
 WHEN JobFamilyGroup like '%POST%' Then 'Post Production'
 ELSE NULL
 END
 AS JOB_FAMILY_GROUP
,CASE
         WHEN DATETRUNC(QUARTER,Date) = '2025-01-01' THEN 'Q1''25'                                                                           
         WHEN DATETRUNC(QUARTER,Date) = '2025-04-01' THEN 'Q2''25'
         WHEN DATETRUNC(QUARTER,Date) = '2025-07-01' THEN 'Q3''25'
         WHEN DATETRUNC(QUARTER,Date) = '2025-10-01' THEN 'Q4''25'                                                                          
         END AS QUARTER
, NULL AS MONTH
,ROUND(SUM(OvertimeHours),0) as TOTAL_OVERTIME 
from opsgroup.ic.ExpectedvsPaidHours
WHERE JobFamilyGroup in ('RECONDITIONING','POST_PRODUCTION')
AND Date >='2025-01-01'
AND Date < '2026-01-01'
GROUP BY LocationName,DATETRUNC(QUARTER,Date),JobFamilyGroup

UNION ALL

-- v3: 2026+ monthly data with dynamic Month/Quarter (SQL Server syntax)
Select LocationName as LOCATION,
DATETRUNC(QUARTER,Date) as DATE_QUARTER,
CASE 
 WHEN JobFamilyGroup like '%RECON%' Then 'Reconditioning'
 WHEN JobFamilyGroup like '%POST%' Then 'Post Production'
 ELSE NULL
 END
 AS JOB_FAMILY_GROUP
,CONCAT('Q', DATEPART(QUARTER, Date), '''', RIGHT(DATEPART(YEAR, Date), 2)) AS QUARTER
,FORMAT(Date, 'yyyy-MM') AS MONTH
,ROUND(SUM(OvertimeHours),0) as TOTAL_OVERTIME 
from opsgroup.ic.ExpectedvsPaidHours
WHERE JobFamilyGroup in ('RECONDITIONING','POST_PRODUCTION')
AND Date >= '2026-01-01'
GROUP BY LocationName,DATETRUNC(QUARTER,Date),JobFamilyGroup,FORMAT(Date, 'yyyy-MM')"""


overtime_sql = pd.read_sql_query(overtimequery, odbcconnops)

In [ ]:
overtime_sql.head()

In [ ]:
overtime_snow = overtime_sql.copy()

In [ ]:
overtime_snow.rename(columns={'LOCATION':'Location','JOB_FAMILY_GROUP':'JobFamilyGroup','QUARTER':'Quarter','TOTAL_OVERTIME':'Total_Overtime_hours_Quarter'},inplace=True)

In [ ]:
# v3: Add Month to overtime before merge
if 'Month' not in overtime_snow.columns:
    overtime_snow['Month'] = None
main_df=pd.merge(main_df,overtime_snow[['Location','JobFamilyGroup','Quarter','Month','Total_Overtime_hours_Quarter']],on=['Location','JobFamilyGroup','Quarter','Month'],how='left')

# Snowflake - Main DF

In [ ]:
# # Create Table
# db_snowflake_write.create_table('metrics',
#                   {
#                      'Location': ['VARCHAR(1000)'],
#                      'Score': ['INT'],
#                      'eSat Change': ['INT'], 
#                      'JobFamilyGroup': ['VARCHAR(1000)'],
#                      'Quarter': ['VARCHAR(1000)'], 
#                      'Injuries': ['INT'],
#                      'EmployeesPIPs_CARs': ['INT'],
#                      'CARs': ['INT'],
#                      'PIPs': ['INT'],
#                      'PIPs & CARs': ['INT'],
#                      'Terminations': ['INT'],
#                      'Quaterly_Turnover': ['INT'],
#                      'Turnover_Change':['INT'],
#                      'Total_Employees': ['INT'],
#                   })

In [ ]:
# main_df[main_df['Quarter']=='Q3\'25'][['Location','JobFamilyGroup','Score','Employee_Size_Peakon','Avg_emp']]

In [ ]:
db_snowflake_write.drop_table('metrics')
db_snowflake_write.dump('metrics',main_df)

In [ ]:
# Inserting the main df
# db_snowflake_write.insert('metrics',main_df,truncate=True)

In [ ]:
# db_snowflake_write.drop_table('main_df')